# HS5652 · Notebook 1 — Multi-Language Data Extraction & Corpus Compilation (Hindi & Telugu)

**Project:** *Cross-Linguistic Computational Analysis of Copula Deletion: A Cross-Register Investigation of Indo-Aryan & Dravidian Languages*  
**Course:** Language, Cognition & Computation (HS5652) | **Team:** Sairam, Varsha, Arya, Nitish, Swapnil

This notebook contains the **complete Python data extraction and corpus compilation code** in the exact execution order used in our project:

| Part | Step | What it does | Original Script / Source | Output CSV(s) |
|---|---|---|---|---|
| **Part I: Hindi** | **Step 1. Subtitle NLP Extraction** | Parses `.srt` film subtitles (*3 Idiots*, *Secret Superstar*, *Taare Zameen Par*, *Mirzapur*), applies Python `set.isdisjoint()` & `str.endswith()` tense filters, and runs batched **Stanza (`hi`)** dependency parsing (plus rule-based baseline comparison) | `setup_stanza.py`, `subtitles_nlp.py`, `subtitles_base.py` | `data/hindi/extracted/extracted_subtitles_nlp*.csv`, `model_comparison_subtitles_*.csv` |
| **Part I: Hindi** | **Step 2. Literary Extraction** | Parses UTF-8 Devanagari short stories (*Idgah*, *Do Bailon Ki Katha*, *Pahalwan Ki Dholak*, *Poos Ki Raat*), separates **Literary Dialogue** vs **Literary Narrative**, filters present tense, and tags with Stanza (`hi`) | `literary.py` | `data/hindi/extracted/formal_*.csv` |
| **Part I: Hindi** | **Step 3. Hindi Compilation** | Merges all per-source CSVs into **one master CSV** with a standardized schema | `cross_register.py`, `cross_register_hypothesis_verify.py` | `data/hindi/Cross_register.csv` ($N=4,334$ raw), `data/hindi/verification/verification.csv` ($N=1,658$ raw) |
| **Part II: Telugu** | **Step 4. Telugu Copula Inventory & Tagger** | Defines Dravidian Telugu copula paradigms (`ఉండు`, `కాదు`, `లేదు`, granthika `కలదు/గలదు/కద్దు`), prothetic glide normalization, verbal suffix blockers, and batched **Stanza (`te`)** clause annotator | `assignment_telugu_copula_deletion.ipynb` (Cells 1–4) | Tagger functions in memory |
| **Part II: Telugu** | **Step 5. Telugu Subtitle Extraction (ASR)** | Downloads/parses word-timed YouTube Telugu auto-captions (`.te-orig.json3`) for 3 Telugu films (*Ala Vaikunthapurramuloo*, *Srirastu Subhamastu*, *Vivaha Bhojanambu*), drops `[సంగీతం]` music stretches, segments utterances at pauses $>0.9\text{ s}$ and verb-final boundaries, and runs Stanza (`te`) | `assignment_telugu_copula_deletion.ipynb` (Cells 5–6) | `subtitles_ala_vaikunthapurramuloo.csv`, `subtitles_srirastu_subhamastu.csv`, `subtitles_vivaha_bhojanambu.csv` |
| **Part II: Telugu** | **Step 6. Telugu Literary Extraction** | Downloads/parses 3 public-domain short stories by Gurajada Apparao (*Diddubatu*, *Mee Peremiti?*, *Metilda*) from `te.wikisource.org`, tracks quotation state across sentence boundaries (*Literary Dialogue* vs *Literary Narrative*), and tags with Stanza (`te`) | `assignment_telugu_copula_deletion.ipynb` (Cells 7–8) | `formal_diddubatu.csv`, `formal_mee_peremiti.csv`, `formal_metilda.csv` |
| **Part II: Telugu** | **Step 7. Telugu Compilation** | Compiles all 6 Telugu sources into the master automatic dataset and loads the manually verified gold set | `assignment_telugu_copula_deletion.ipynb` (Cells 9–10) | `data/telugu/Telugu_copula_dataset_automatic.csv` ($N=759$), `data/telugu/Telugu_copula_dataset_manual.csv` ($N=116$) |
| **Part III: Overview** | **Step 8. All-5-Language Dataset Summary** | Verifies all compiled and curated datasets across **Hindi, Assamese, Bengali, Telugu, and Tamil** ready for Notebook 2 (`02_cross_linguistic_analysis.ipynb`) | Cross-check | Summary table |

> **Reproducibility Note:** Heavy neural parsing steps (`stanza`, `pysrt`, `indic-nlp-library`, `yt-dlp`) require raw `.srt`/`.json3` files. By default `RUN_HINDI_EXTRACTION = False` and `RUN_TELUGU_EXTRACTION = False` so that all extraction functions are defined in full and the pre-extracted CSVs in `data/` are loaded and compiled end-to-end. Set either flag to `True` to re-run Stanza neural extraction from raw files.

In [1]:
# ---------------------------------------------------------------
# 0. Setup: paths, flags and common imports
# ---------------------------------------------------------------
# Optional (only needed when RUN_HINDI_EXTRACTION or RUN_TELUGU_EXTRACTION = True):
#   !pip install stanza pysrt indic-nlp-library yt-dlp beautifulsoup4 requests
import os
import re
import json
import glob
import string
import warnings
import pandas as pd

warnings.filterwarnings("ignore")

RUN_HINDI_EXTRACTION  = False   # True -> re-run Hindi Stanza extraction from raw .srt / .txt files
RUN_TELUGU_EXTRACTION = False   # True -> re-run Telugu download & Stanza extraction from Wikisource / YouTube json3

BASE_DIR         = os.getcwd()  # run from inside 'materials' folder
HINDI_DIR        = os.path.join(BASE_DIR, "data", "hindi")
RAW_TEXT_DIR     = os.path.join(HINDI_DIR, "raw_texts")      # Hindi literary .txt stories
EXTRACTED_DIR    = os.path.join(HINDI_DIR, "extracted")      # Hindi per-source extraction CSVs
SRT_DIR          = os.path.join(HINDI_DIR, "srt")            # Put original .srt files here to re-run Step 1
TELUGU_DIR       = os.path.join(BASE_DIR, "data", "telugu")  # Telugu datasets folder

for d in (HINDI_DIR, RAW_TEXT_DIR, EXTRACTED_DIR, TELUGU_DIR):
    assert os.path.isdir(d), f"Missing folder: {d}"

print("Working directory        :", BASE_DIR)
print("Hindi extraction mode    :", "RE-RUN from raw files" if RUN_HINDI_EXTRACTION else "load pre-extracted CSVs & compile")
print("Telugu extraction mode   :", "RE-RUN from raw files" if RUN_TELUGU_EXTRACTION else "load pre-extracted CSVs & verify")

Working directory        : /home/antigravity/nitishgupt100/language_cognition_computation/materials
Hindi extraction mode    : load pre-extracted CSVs & compile
Telugu extraction mode   : load pre-extracted CSVs & verify


---
# PART I · Hindi Data Extraction & Compilation

## Step 1 · Hindi Subtitle NLP Extraction (Spoken Register)

**Sources:** *3 Idiots*, *Secret Superstar*, *Taare Zameen Par* (main 6-source corpus) and *Mirzapur* (held-out verification set).

**Pipeline (`setup_stanza.py`, `subtitles_nlp.py`, `subtitles_base.py`):**
1. `pysrt` reads the `.srt` subtitle file and strips formatting tags.
2. `indicnlp.tokenize.sentence_tokenize` splits the text into Hindi Devanagari sentences.
3. **Fast lexical pre-filter** with Python `set.isdisjoint()` — drops fragments ($< 3$ words), past-tense markers (`था/थी/गया…`), future auxiliaries (`होगा…`), and future suffixes (`-एगा/-एंगे…`) via `str.endswith()`.
4. **Batched Stanza parsing** (`tokenize,pos,lemma,depparse`) of all remaining present-tense candidates.
5. **Tagging:**
   - `Copula_Status`: `1` if present-tense copula (`है/हैं/हूँ/हुं/हो`) is present, else `0` (deleted/zero).
   - `Polarity`: `Negative` (`neg` deprel or `नहीं/न/ना/मत/बिना/बगैर`), else `Affirmative`.
   - `Predicate_Type_Final`: from Universal Dependencies root (`ADJ` $\rightarrow$ `Adjectival`; `NOUN/PROPN/PRON` $\rightarrow$ `Equational/Nominal`; locative postposition / `obl` / `ADP` dependent $\rightarrow$ `Locative/Existential`).

In [2]:
# ---------------------------------------------------------------
# 1a. Hindi Stanza model setup (one-time download) [setup_stanza.py]
# ---------------------------------------------------------------
_NLP_HI = None

def get_hindi_stanza_pipeline():
    """Lazily initialise the Hindi Stanza pipeline ONCE (no repeated downloads)."""
    global _NLP_HI
    if _NLP_HI is None:
        import stanza
        print("Downloading / loading Hindi Stanza model (tokenize,pos,lemma,depparse)...")
        stanza.download('hi', processors='tokenize,pos,lemma,depparse', verbose=False)
        _NLP_HI = stanza.Pipeline('hi', processors='tokenize,pos,lemma,depparse', verbose=False, use_gpu=False)
    return _NLP_HI


# ---------------------------------------------------------------
# 1b. Hindi Subtitle NLP Extraction [subtitles_nlp.py]
# ---------------------------------------------------------------
def process_hindi_subtitles_fast(srt_file_path, output_csv_path="extracted_subtitles_nlp4.csv", movie_title="Movie_01"):
    import pysrt
    import stanza
    from indicnlp.tokenize import sentence_tokenize
    nlp = get_hindi_stanza_pipeline()

    # 2. Parse SRT file using pysrt
    subs = pysrt.open(srt_file_path, encoding='utf-8')
    clean_lines = [sub.text_without_tags.strip() for sub in subs if sub.text_without_tags.strip()]
    full_text = " ".join(clean_lines)

    # 3. Split into sentences using Indic NLP Library
    raw_sentences = sentence_tokenize.sentence_split(full_text, lang='hi')

    # Linguistic Marker Sets for Fast Filtering
    present_copulas = {"है", "हैं", "हूँ", "हुं", "हो"}
    past_markers = {"था", "थी", "थे", "थीं", "हुआ", "हुई", "हुए", "हुईं", "गया", "गयी", "गये", "गई"}
    future_auxiliaries = {"होगा", "होगी", "होंगे", "होगीं", "होऊंगा", "होओगे"}
    negative_markers = {"नहीं", "न", "ना", "मत", "बिना", "बगैर"}
    locative_postpositions = {
        "में", "पर", "पे", "तक", "के पास", "के अंदर", "के भीतर",
        "के बाहर", "के ऊपर", "के नीचे", "के सामने", "के पीछे",
        "के बीच", "के यहाँ", "की तरफ", "की ओर"
    }

    hindi_punctuation = string.punctuation + "।॥''\"\"“”"

    filtered_candidates = []

    print("Pre-filtering present-tense candidates...")
    for raw in raw_sentences:
        sentence = raw.strip()
        raw_words = sentence.split()
        words = [w.strip(hindi_punctuation) for w in raw_words if w.strip(hindi_punctuation)]

        if len(words) < 3:
            continue

        word_set = set(words)

        if not word_set.isdisjoint(past_markers) or not word_set.isdisjoint(future_auxiliaries):
            continue
        if any(w.endswith(("एगा", "एगी", "एंगे", "ेंगी", "ऊंगा", "ओगे")) for w in words):
            continue

        filtered_candidates.append((sentence, word_set))

    print(f"Found {len(filtered_candidates)} candidate present-tense sentences. Running batched Stanza NLP...")

    # 4. Prepare Stanza Documents for BATCH PROCESSING
    in_docs = [stanza.Document([], text=item[0]) for item in filtered_candidates]
    out_docs = nlp(in_docs)  # Runs all neural parsing in a single batched operation!

    annotated_rows = []

    for sentence_id, ((sentence, word_set), doc) in enumerate(zip(filtered_candidates, out_docs), start=1):
        copula_status = 1 if not word_set.isdisjoint(present_copulas) else 0

        # Polarity
        is_negative = False
        for sent in doc.sentences:
            for token in sent.words:
                if token.deprel == 'neg' or token.text in negative_markers:
                    is_negative = True
                    break
        polarity = "Negative" if is_negative else "Affirmative"

        # Predicate Classification
        root_token = None
        for sent in doc.sentences:
            for token in sent.words:
                if token.deprel == 'root':
                    root_token = token
                    break

        is_locative = any(postpos in sentence for postpos in locative_postpositions)

        if root_token:
            if is_locative or root_token.deprel == 'obl' or any(w.upos == 'ADP' for sent in doc.sentences for w in sent.words if w.head == root_token.id):
                predicate_type = "Locative/Existential"
            elif root_token.upos == 'ADJ':
                predicate_type = "Adjectival"
            elif root_token.upos in ['NOUN', 'PROPN', 'PRON']:
                predicate_type = "Equational/Nominal"
            elif root_token.upos == 'VERB':
                predicate_type = "Adjectival" if any(w.upos == 'ADJ' for sent in doc.sentences for w in sent.words) else "Equational/Nominal"
            else:
                predicate_type = "Equational/Nominal"
        else:
            predicate_type = "Locative/Existential" if is_locative else "Equational/Nominal"

        annotated_rows.append({
            "Sentence_ID": f"SUB_{sentence_id:03d}",
            "Register": "Subtitle",
            "Movie_Title": movie_title,
            "Raw_Sentence": sentence,
            "Copula_Status": copula_status,
            "Polarity": polarity,
            "Predicate_Type_Hint": f"NLP Parsed ({root_token.upos if root_token else 'UNK'})",
            "Predicate_Type_Final": predicate_type,
            "Notes": f"Root Word: {root_token.text if root_token else 'N/A'}"
        })

    df = pd.DataFrame(annotated_rows)
    df.to_csv(output_csv_path, index=False, encoding='utf-8-sig')

    print(f"Extraction complete! Saved {len(df)} tagged sentences to '{output_csv_path}'.")
    return df

In [3]:
# ---------------------------------------------------------------
# 1c. Rule-based baseline extractor (for Base-vs-NLP comparison) [subtitles_base.py]
# ---------------------------------------------------------------
def process_hindi_subtitles_pysrt(srt_file_path, output_csv_path="extracted_subtitles3.csv", movie_title="Movie_01"):
    """Parses an SRT file, strips punctuation, excludes past/future tenses using expanded
    marker sets, auto-detects present copula status, polarity, and locative heuristics."""
    import pysrt
    from indicnlp.tokenize import sentence_tokenize

    subs = pysrt.open(srt_file_path, encoding='utf-8')
    clean_lines = [sub.text_without_tags.strip() for sub in subs if sub.text_without_tags.strip()]
    full_text = " ".join(clean_lines)

    raw_sentences = sentence_tokenize.sentence_split(full_text, lang='hi')

    present_copulas = {"है", "हैं", "हूँ", "हुं", "हो"}
    past_markers = {"था", "थी", "थे", "थीं", "हुआ", "हुई", "हुए", "हुईं", "गया", "गयी", "गये", "गई"}
    future_auxiliaries = {"होगा", "होगी", "होंगे", "होगीं", "होऊंगा", "होओगे"}
    negative_markers = {"नहीं", "न", "ना", "मत", "बिना", "बगैर"}
    locative_postpositions = [
        "में", "पर", "पे", "तक", "के पास", "के अंदर", "के भीतर",
        "के बाहर", "के ऊपर", "के नीचे", "के सामने", "के पीछे",
        "के बीच", "के यहाँ", "की तरफ", "की ओर"
    ]

    hindi_punctuation = string.punctuation + "।॥''\"\"“”"

    annotated_rows = []
    sentence_id = 1

    for raw in raw_sentences:
        sentence = raw.strip()
        raw_words = sentence.split()
        words = [w.strip(hindi_punctuation) for w in raw_words if w.strip(hindi_punctuation)]

        if len(words) < 3:
            continue

        word_set = set(words)

        if not word_set.isdisjoint(past_markers):
            continue
        if not word_set.isdisjoint(future_auxiliaries):
            continue
        if any(w.endswith(("एगा", "एगी", "एंगे", "ेंगी", "ऊंगा", "ओगे")) for w in words):
            continue

        copula_status = 1 if not word_set.isdisjoint(present_copulas) else 0
        polarity = "Negative" if not word_set.isdisjoint(negative_markers) else "Affirmative"
        is_locative = any(postpos in sentence for postpos in locative_postpositions)
        predicate_hint = "Locative/Existential" if is_locative else "Nominal/Adjectival (Check Manually)"

        annotated_rows.append({
            "Sentence_ID": f"SUB_{sentence_id:03d}",
            "Register": "Subtitle",
            "Movie_Title": movie_title,
            "Raw_Sentence": sentence,
            "Copula_Status": copula_status,
            "Polarity": polarity,
            "Predicate_Type_Hint": predicate_hint,
            "Predicate_Type_Final": "",
            "Notes": ""
        })
        sentence_id += 1

    df = pd.DataFrame(annotated_rows)
    df.to_csv(output_csv_path, index=False, encoding='utf-8-sig')
    print(f"Extraction complete! Processed {len(df)} candidate sentences into '{output_csv_path}'.")
    return df


def build_model_comparison(base_csv, nlp_csv, output_csv):
    """Aligns the rule-based (Base) and Stanza (NLP) outputs row-by-row and records agreement."""
    B = pd.read_csv(base_csv, encoding='utf-8-sig')
    N = pd.read_csv(nlp_csv, encoding='utf-8-sig')
    assert (B['Raw_Sentence'] == N['Raw_Sentence']).all(), "Base and NLP files are not aligned"
    out = pd.DataFrame({
        "Sentence_ID": B["Sentence_ID"], "Register": B["Register"],
        "Movie_Title": B["Movie_Title"], "Raw_Sentence": B["Raw_Sentence"],
        "Copula_Status_Base": B["Copula_Status"], "Copula_Status_NLP": N["Copula_Status"],
        "Copula_Status_Match": B["Copula_Status"] == N["Copula_Status"],
        "Polarity_Base": B["Polarity"], "Polarity_NLP": N["Polarity"],
        "Polarity_Match": B["Polarity"] == N["Polarity"],
        "Predicate_Type_Hint_Base": B["Predicate_Type_Hint"], "Predicate_Type_Hint_NLP": N["Predicate_Type_Hint"],
        "Predicate_Type_Final_Base": B["Predicate_Type_Final"], "Predicate_Type_Final_NLP": N["Predicate_Type_Final"],
        "NLP_Root_Word_Notes": N["Notes"],
    })
    out.to_csv(output_csv, index=False, encoding='utf-8-sig')
    return out

In [4]:
# ---------------------------------------------------------------
# 1d. Run Step 1 for every Hindi film (or load pre-extracted CSVs)
# ---------------------------------------------------------------
SUBTITLE_JOBS = [
    ("3 Idiots",         "3_Idiots.hin.srt",                 "extracted_subtitles.csv",  "extracted_subtitles_nlp.csv",  "model_comparison_subtitles_3_idiots.csv"),
    ("Secret_superstar", "Secret_Superstar.hin.srt",         "extracted_subtitles2.csv", "extracted_subtitles_nlp2.csv", "model_comparison_subtitles_secret_superstar.csv"),
    ("Taare_zameen_par", "TaareZameenPar.SHUDDH.Hindi.srt",  "extracted_subtitles3.csv", "extracted_subtitles_nlp3.csv", "model_comparison_subtitles_taare_zameen_par.csv"),
    ("Mirzapur",         "Mirzapur.The.Movie.2026.hin.srt",  None,                       "extracted_subtitles_nlp4.csv", None),   # held-out verification film
]

subtitle_summary = []
for title, srt_name, base_csv, nlp_csv, cmp_csv in SUBTITLE_JOBS:
    srt_path = os.path.join(SRT_DIR, srt_name)
    nlp_path = os.path.join(EXTRACTED_DIR, nlp_csv)
    if RUN_HINDI_EXTRACTION and os.path.exists(srt_path):
        df_nlp = process_hindi_subtitles_fast(srt_path, nlp_path, title)
        if base_csv:
            process_hindi_subtitles_pysrt(srt_path, os.path.join(EXTRACTED_DIR, base_csv), title)
    else:
        df_nlp = pd.read_csv(nlp_path, encoding='utf-8-sig')
    if base_csv and cmp_csv:
        cmp = build_model_comparison(os.path.join(EXTRACTED_DIR, base_csv), nlp_path, os.path.join(EXTRACTED_DIR, cmp_csv))
        cop_agree = cmp["Copula_Status_Match"].mean() * 100
        pol_agree = cmp["Polarity_Match"].mean() * 100
    else:
        cop_agree = pol_agree = float("nan")
    subtitle_summary.append({
        "Film": title,
        "Candidate_Sentences": len(df_nlp),
        "Copula_Present": int((df_nlp["Copula_Status"] == 1).sum()),
        "Copula_Absent": int((df_nlp["Copula_Status"] == 0).sum()),
        "Base_vs_NLP_Copula_Agreement_%": round(cop_agree, 1),
        "Base_vs_NLP_Polarity_Agreement_%": round(pol_agree, 1),
    })

print("STEP 1 SUMMARY — Hindi Spoken Register (Film Subtitles)")
display(pd.DataFrame(subtitle_summary))
print("\nSample rows from 'extracted_subtitles_nlp.csv' (3 Idiots):")
display(pd.read_csv(os.path.join(EXTRACTED_DIR, "extracted_subtitles_nlp.csv"), encoding='utf-8-sig').iloc[1:7])

STEP 1 SUMMARY — Hindi Spoken Register (Film Subtitles)


,Film,Candidate_Sentences,Copula_Present,Copula_Absent,Base_vs_NLP_Copula_Agreement_%,Base_vs_NLP_Polarity_Agreement_%
0,3 Idiots,1373,611,762,100.0,99.3
1,Secret_superstar,1401,690,711,100.0,99.1
2,Taare_zameen_par,843,411,432,100.0,99.5
3,Mirzapur,1555,760,795,NaN,NaN



Sample rows from 'extracted_subtitles_nlp.csv' (3 Idiots):


,Sentence_ID,Register,Movie_Title,Raw_Sentence,Copula_Status,Polarity,Predicate_Type_Hint,Predicate_Type_Final,Notes
1,SUB_002,Subtitle,3 Idiots,"सिर, आपका मोबाइल क्रपाय बंद कीजिए एक सेकंड प्ल...",0,Affirmative,NLP Parsed (VERB),Adjectival,Root Word: कीजिए
2,SUB_003,Subtitle,3 Idiots,"सिर, बैटिए प्लीज।",0,Affirmative,NLP Parsed (VERB),Equational/Nominal,Root Word: प्लीज
3,SUB_004,Subtitle,3 Idiots,नहीं मैं बिल्कुल ठीक हूँ।,1,Negative,NLP Parsed (ADJ),Adjectival,Root Word: ठीक
4,SUB_005,Subtitle,3 Idiots,"आप लोग जाएं, आप लोग जाएं।",0,Affirmative,NLP Parsed (VERB),Equational/Nominal,Root Word: जाएं
5,SUB_006,Subtitle,3 Idiots,इंतजार करो गाड़ी निकालो यार।,0,Affirmative,NLP Parsed (VERB),Equational/Nominal,Root Word: निकालो
6,SUB_007,Subtitle,3 Idiots,तुस्सी मि. ढिल्लों?,0,Affirmative,NLP Parsed (PROPN),Equational/Nominal,Root Word: ढिल्लों


---
## Step 2 · Hindi Literary Extraction (Formal Register)

**Sources:** *Idgah* (Premchand), *Do Bailon Ki Katha* (Premchand), *Pahalwan Ki Dholak* (Phanishwar Nath Renu) for the main corpus, and *Poos Ki Raat* (Premchand) for the held-out verification corpus.

**Pipeline (`literary.py`):**
1. Reads UTF-8 Devanagari `.txt` story files from `data/hindi/raw_texts/`.
2. Uses `indicnlp.tokenize.sentence_tokenize` to split prose into sentences.
3. Detects direct character speech via quotation regex (`["'“«].*?["'”»]`) to separate **Literary Dialogue** from **Literary Narrative**.
4. Applies present-tense filtering and runs batched **Stanza (`hi`)** dependency parsing.

In [5]:
# ---------------------------------------------------------------
# 2a. Hindi Literary Extraction [literary.py]
# ---------------------------------------------------------------
def process_hindi_literary_txt(txt_path, output_csv_path="formal_idgah.csv", story_title="idgah"):
    """Parses a UTF-8 encoded Hindi literary text file (.txt), separates direct character dialogue
    from authorial narrative prose, filters present-tense sentences, and uses Stanza NLP
    to tag Copula Status, Polarity, and Predicate Types for formal register analysis."""
    import stanza
    from indicnlp.tokenize import sentence_tokenize

    if not os.path.exists(txt_path):
        raise FileNotFoundError(f"Text file not found at: '{txt_path}'")

    print(f"Step 1: Reading clean Devanagari text from '{txt_path}'...")
    with open(txt_path, 'r', encoding='utf-8') as f:
        full_text = f.read()

    nlp = get_hindi_stanza_pipeline()
    raw_sentences = sentence_tokenize.sentence_split(full_text, lang='hi')

    present_copulas = {"है", "हैं", "हूँ", "हुं", "हो"}
    past_markers = {"था", "थी", "थे", "थीं", "हुआ", "हुई", "हुए", "हुईं", "गया", "गयी", "गये", "गई"}
    future_auxiliaries = {"होगा", "होगी", "होंगे", "होगीं", "होऊंगा", "होओगे"}
    negative_markers = {"नहीं", "न", "ना", "मत", "बिना", "बगैर"}
    locative_postpositions = {
        "में", "पर", "पे", "तक", "के पास", "के अंदर", "के भीतर",
        "के बाहर", "के ऊपर", "के नीचे", "के सामने", "के पीछे",
        "के बीच", "के यहाँ", "की तरफ", "की ओर"
    }

    hindi_punctuation = string.punctuation + "।॥''\"\"“”«»"
    filtered_candidates = []

    print("Step 3: Filtering present-tense sentences & categorizing dialogue vs narrative...")
    for raw in raw_sentences:
        sentence = raw.strip()
        is_dialogue = bool(re.search(r'["\'“«].*?["\'”»]', sentence)) or sentence.startswith(('"', "'", "“", "«"))

        raw_words = sentence.split()
        words = [w.strip(hindi_punctuation) for w in raw_words if w.strip(hindi_punctuation)]

        if len(words) < 3:
            continue

        word_set = set(words)

        if not word_set.isdisjoint(past_markers) or not word_set.isdisjoint(future_auxiliaries):
            continue
        if any(w.endswith(("एगा", "एगी", "एंगे", "ेंगी", "ऊंगा", "ओगे")) for w in words):
            continue

        filtered_candidates.append((sentence, word_set, is_dialogue))

    columns = [
        "Sentence_ID", "Register", "Sub_Register", "Movie_Title", "Raw_Sentence",
        "Copula_Status_Base", "Copula_Status_NLP", "Copula_Status_Match",
        "Polarity_Base", "Polarity_NLP", "Polarity_Match",
        "Predicate_Type_Hint_Base", "Predicate_Type_Hint_NLP",
        "Predicate_Type_Final_Base", "Predicate_Type_Final_NLP", "NLP_Root_Word_Notes"
    ]

    if not filtered_candidates:
        df = pd.DataFrame(columns=columns)
        df.to_csv(output_csv_path, index=False, encoding='utf-8-sig')
        return df

    in_docs = [stanza.Document([], text=item[0]) for item in filtered_candidates]
    out_docs = nlp(in_docs)

    annotated_rows = []
    for sentence_id, ((sentence, word_set, is_dialogue), doc) in enumerate(zip(filtered_candidates, out_docs), start=1):
        copula_status = 1 if not word_set.isdisjoint(present_copulas) else 0

        is_negative = any(
            token.deprel == 'neg' or token.text in negative_markers
            for sent in doc.sentences for token in sent.words
        )
        polarity = "Negative" if is_negative else "Affirmative"

        root_token = None
        for sent in doc.sentences:
            for token in sent.words:
                if token.deprel == 'root':
                    root_token = token
                    break

        is_locative = any(postpos in sentence for postpos in locative_postpositions)

        if root_token:
            if is_locative or root_token.deprel == 'obl' or any(w.upos == 'ADP' for sent in doc.sentences for w in sent.words if w.head == root_token.id):
                predicate_type = "Locative/Existential"
            elif root_token.upos == 'ADJ':
                predicate_type = "Adjectival"
            elif root_token.upos in ['NOUN', 'PROPN', 'PRON']:
                predicate_type = "Equational/Nominal"
            elif root_token.upos == 'VERB':
                predicate_type = "Adjectival" if any(w.upos == 'ADJ' for sent in doc.sentences for w in sent.words) else "Equational/Nominal"
            else:
                predicate_type = "Equational/Nominal"
        else:
            predicate_type = "Locative/Existential" if is_locative else "Equational/Nominal"

        text_sub_type = "Literary Dialogue" if is_dialogue else "Literary Narrative"

        annotated_rows.append({
            "Sentence_ID": f"LIT_{sentence_id:04d}",
            "Register": "Literary",
            "Sub_Register": text_sub_type,
            "Movie_Title": story_title,
            "Raw_Sentence": sentence,
            "Copula_Status_Base": copula_status,
            "Copula_Status_NLP": copula_status,
            "Copula_Status_Match": True,
            "Polarity_Base": "Negative" if not word_set.isdisjoint(negative_markers) else "Affirmative",
            "Polarity_NLP": polarity,
            "Polarity_Match": ("Negative" if not word_set.isdisjoint(negative_markers) else "Affirmative") == polarity,
            "Predicate_Type_Hint_Base": "Locative/Existential" if is_locative else "Nominal/Adjectival (Check Manually)",
            "Predicate_Type_Hint_NLP": f"NLP Parsed ({root_token.upos if root_token else 'UNK'})",
            "Predicate_Type_Final_Base": "",
            "Predicate_Type_Final_NLP": predicate_type,
            "NLP_Root_Word_Notes": f"Root Word: {root_token.text if root_token else 'N/A'}"
        })

    df = pd.DataFrame(annotated_rows, columns=columns)
    df.to_csv(output_csv_path, index=False, encoding='utf-8-sig')
    return df


# ---------------------------------------------------------------
# 2b. Run Step 2 for every Hindi short story (or load pre-extracted CSVs)
# ---------------------------------------------------------------
LITERARY_JOBS = [
    ("idgah",                "idgah.txt",               "formal_idgah.csv"),
    ("do_bailon_ki_katha",   "katha.txt",               "formal_katha.csv"),
    ("pahalwan_ki_dholak",   "pahalwan_ki_dholak.txt",  "formal_pahalwa_ki_dholak.csv"),
    ("poos_ki_raat",         "poos_ki_raat.txt",        "formal_poos_ki_raat.csv"),     # held-out verification story
]

literary_summary = []
for title, txt_name, out_csv in LITERARY_JOBS:
    txt_path = os.path.join(RAW_TEXT_DIR, txt_name)
    out_path = os.path.join(EXTRACTED_DIR, out_csv)
    if RUN_HINDI_EXTRACTION:
        df_lit = process_hindi_literary_txt(txt_path, out_path, title)
    else:
        df_lit = pd.read_csv(out_path, encoding='utf-8-sig')
    literary_summary.append({
        "Story": title,
        "Raw_Text_Chars": len(open(txt_path, encoding='utf-8').read()),
        "Candidate_Sentences": len(df_lit),
        "Literary_Narrative": int((df_lit["Sub_Register"] == "Literary Narrative").sum()),
        "Literary_Dialogue": int((df_lit["Sub_Register"] == "Literary Dialogue").sum()),
        "Copula_Present": int((df_lit["Copula_Status_NLP"] == 1).sum()),
        "Copula_Absent": int((df_lit["Copula_Status_NLP"] == 0).sum()),
    })

print("STEP 2 SUMMARY — Hindi Formal Register (Literary Fiction)")
display(pd.DataFrame(literary_summary))

STEP 2 SUMMARY — Hindi Formal Register (Literary Fiction)


,Story,Raw_Text_Chars,Candidate_Sentences,Literary_Narrative,Literary_Dialogue,Copula_Present,Copula_Absent
0,idgah,23545,357,346,11,182,175
1,do_bailon_ki_katha,20070,246,187,59,89,157
2,pahalwan_ki_dholak,13721,114,114,0,15,99
3,poos_ki_raat,9645,103,99,4,28,75


---
## Step 3 · Hindi Master Corpus Compilation (`cross_register.py`)

Combines all per-source Hindi CSV files into a single master CSV (`data/hindi/Cross_register.csv` for the 6 main sources; `data/hindi/verification/verification.csv` for the 2 held-out verification sources).

In [6]:
# ---------------------------------------------------------------
# 3a. Compile the 6-source Hindi master corpus [cross_register.py]
# ---------------------------------------------------------------
COLUMNS_TO_KEEP = [
    "Sentence_ID", "Register", "Sub_Register", "Source_Title", "Raw_Sentence",
    "Copula_Status_NLP", "Polarity_NLP", "Predicate_Type_Final_NLP", "NLP_Root_Word_Notes"
]
COL_RENAME = {
    'Copula_Status': 'Copula_Status_NLP',
    'Polarity': 'Polarity_NLP',
    'Predicate_Type_Final': 'Predicate_Type_Final_NLP',
    'Notes': 'NLP_Root_Word_Notes'
}

def compile_hindi_corpus(file_mapping, output_path, id_prefix):
    master_dfs = []
    for title, (filepath, register, sub_reg) in file_mapping.items():
        if not os.path.exists(filepath):
            print(f"Skipping missing file: {filepath}")
            continue

        df = pd.read_csv(filepath, encoding='utf-8-sig')
        df['Source_Title'] = title
        df['Register'] = register
        if 'Sub_Register' not in df.columns:
            df['Sub_Register'] = sub_reg
        df = df.rename(columns={k: v for k, v in COL_RENAME.items() if v not in df.columns})

        existing_cols = [c for c in COLUMNS_TO_KEEP if c in df.columns]
        master_dfs.append(df[existing_cols])

    if not master_dfs:
        raise RuntimeError("Error: No valid CSV files were loaded.")

    master_corpus = pd.concat(master_dfs, ignore_index=True)
    master_corpus['Master_Sentence_ID'] = [f"{id_prefix}_{i:04d}" for i in range(1, len(master_corpus) + 1)]

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    master_corpus.to_csv(output_path, index=False, encoding='utf-8-sig')

    print("=" * 65)
    print(f"SUCCESS: Hindi master corpus created with {len(master_corpus)} rows.")
    print(f"Saved at: '{os.path.relpath(output_path, BASE_DIR)}'")
    return master_corpus


E = lambda f: os.path.join(EXTRACTED_DIR, f)
hindi_main_mapping = {
    "3_Idiots":           (E("model_comparison_subtitles_3_idiots.csv"),          "Spoken Subtitle", "Film Dialogue"),
    "Secret_Superstar":   (E("model_comparison_subtitles_secret_superstar.csv"),  "Spoken Subtitle", "Film Dialogue"),
    "Taare_Zameen_Par":   (E("model_comparison_subtitles_taare_zameen_par.csv"),  "Spoken Subtitle", "Film Dialogue"),
    "Idgah":              (E("formal_idgah.csv"),                                 "Literary", "Literary Narrative / Dialogue"),
    "Pahlwan_Ki_Dholak":  (E("formal_pahalwa_ki_dholak.csv"),                     "Literary", "Literary Narrative / Dialogue"),
    "Do_Bailon_Ki_Katha": (E("formal_katha.csv"),                                 "Literary", "Literary Narrative / Dialogue"),
}

master_corpus = compile_hindi_corpus(hindi_main_mapping, os.path.join(HINDI_DIR, "Cross_register.csv"), "CORPUS")
display(master_corpus.groupby(['Register', 'Sub_Register', 'Source_Title']).size().rename('Rows').reset_index())

# 3b. Compile the held-out Hindi verification corpus [cross_register_hypothesis_verify.py]
verification_mapping = {
    "Mirzapur":     (E("extracted_subtitles_nlp4.csv"), "Spoken Subtitle", "Film Dialogue"),
    "Poos_Ki_Raat": (E("formal_poos_ki_raat.csv"),      "Literary",        "Literary Narrative / Dialogue"),
}
verification_df = compile_hindi_corpus(
    verification_mapping,
    os.path.join(HINDI_DIR, "verification", "verification.csv"),
    "VERIFY"
)

SUCCESS: Hindi master corpus created with 4334 rows.
Saved at: 'data/hindi/Cross_register.csv'


,Register,Sub_Register,Source_Title,Rows
0,Literary,Literary Dialogue,Do_Bailon_Ki_Katha,59
1,Literary,Literary Dialogue,Idgah,11
2,Literary,Literary Narrative,Do_Bailon_Ki_Katha,187
3,Literary,Literary Narrative,Idgah,346
4,Literary,Literary Narrative,Pahlwan_Ki_Dholak,114
5,Spoken Subtitle,Film Dialogue,3_Idiots,1373
6,Spoken Subtitle,Film Dialogue,Secret_Superstar,1401
7,Spoken Subtitle,Film Dialogue,Taare_Zameen_Par,843


SUCCESS: Hindi master corpus created with 1658 rows.
Saved at: 'data/hindi/verification/verification.csv'


---
# PART II · Telugu Data Extraction & Corpus Compilation (`assignment_telugu_copula_deletion.ipynb`)

Why Telugu (South-Central Dravidian) requires dedicated extraction rules compared to Hindi:
- **Equational / Nominal (Affirmative):** Telugu has **no overt present-tense copula** (`వాడు నా తమ్ముడు` 'he [is] my brother' $\rightarrow$ categorical zero copula).
- **Locative / Existential & Adjectival-gā (Affirmative):** Uses `ఉండు` (`ఉంది/ఉన్నాడు/ఉన్నారు…`) or literary granthika `కలదు/గలదు/కద్దు`.
- **Negative Clauses:** Negation is fused into dedicated negative copula verbs — `కాదు`-family for equational negation (`వాడు నా తమ్ముడు కాదు`) and `లేదు`-family for existential/locative negation (`అమ్మ ఇంట్లో లేదు`).

**Data Sources Extracted in Python:**
1. **Spoken Register (YouTube Telugu ASR `json3`):** *Ala Vaikunthapurramuloo* (2020), *Srirastu Subhamastu* (2016), and *Vivaha Bhojanambu* (2021).
2. **Literary Register (Wikisource Public-Domain Fiction):** Three short stories by **Gurajada Apparao** (*Diddubatu* 1910, *Mee Peremiti?* 1910, *Metilda*).

In [7]:
# ---------------------------------------------------------------
# Step 4 · Telugu Raw Data Acquisition & Stanza Copula Tagger
# ---------------------------------------------------------------
def download_telugu_raw_sources(telugu_raw_dir):
    """Downloads Gurajada Apparao short stories from te.wikisource.org and
    YouTube Telugu auto-captions (word-timed json3) using yt-dlp."""
    import subprocess, sys, requests
    from bs4 import BeautifulSoup

    stories_dir = os.path.join(telugu_raw_dir, "stories")
    subs_dir = os.path.join(telugu_raw_dir, "subtitles")
    os.makedirs(stories_dir, exist_ok=True)
    os.makedirs(subs_dir, exist_ok=True)

    # 1. Literary register: Gurajada Apparao short stories (1910-1914) from te.wikisource.org
    WIKISOURCE = {"diddubatu": "దిద్దుబాటు", "mee_peremiti": "మీ పేరేమిటి?", "metilda": "మెటిల్డా"}
    for fn, title in WIKISOURCE.items():
        path = os.path.join(stories_dir, f"{fn}.txt")
        if os.path.exists(path):
            continue
        r = requests.get("https://te.wikisource.org/w/api.php", headers={"User-Agent": "LCC-assignment/1.0"},
                         params={"action": "parse", "page": title, "prop": "text", "format": "json", "redirects": 1}).json()
        soup = BeautifulSoup(r["parse"]["text"]["*"], "html.parser")
        for x in soup.select(".ws-noexport, .noprint, .pagenum, .ws-pagenum, sup.reference, .mw-editsection, table, style"):
            x.decompose()
        paras = [p.get_text(" ", strip=True).replace("\u200b", "") for p in soup.find_all(["p", "dd", "li"])]
        paras = [p for p in paras if p and sum(c.isascii() and c.isalpha() for c in p) < 0.3 * max(1, sum(c.isalpha() for c in p))]
        open(path, "w", encoding="utf-8").write("\n".join(paras))
        print(f"Downloaded {path}")

    # 2. Spoken register: YouTube Telugu auto-captions (word-timed json3)
    YOUTUBE = {
        "ala_vaikunthapurramuloo": "bdqZttmi-pk",
        "srirastu_subhamastu": "Xr03nYjJ8AI",
        "vivaha_bhojanambu": "AwucEgasL64"
    }
    for fn, vid in YOUTUBE.items():
        out_json3 = os.path.join(subs_dir, f"{fn}.te-orig.json3")
        if os.path.exists(out_json3):
            continue
        subprocess.run([
            sys.executable, "-m", "yt_dlp", "--skip-download", "--write-auto-subs",
            "--sub-langs", "te-orig", "--sub-format", "json3",
            "-o", os.path.join(subs_dir, f"{fn}.%(ext)s"),
            f"https://www.youtube.com/watch?v={vid}"
        ], check=False)


# ---------------------------------------------------------------
# TELUGU COPULA INVENTORY, PRESENT-TENSE FILTER & STANZA TAGGER
# ---------------------------------------------------------------
_NLP_TE = None

def get_telugu_stanza_pipeline():
    global _NLP_TE
    if _NLP_TE is None:
        import stanza
        print("Initializing Stanza Telugu pipeline (tokenize,pos,lemma,depparse)...")
        stanza.download('te', processors='tokenize,pos,lemma,depparse', verbose=False)
        _NLP_TE = stanza.Pipeline('te', processors='tokenize,pos,lemma,depparse', verbose=False, use_gpu=False)
    return _NLP_TE

EXIST_AFF = {"ఉంది", "ఉన్నది", "ఉన్నాది", "ఉన్నాడు", "ఉన్నాడ్", "ఉన్నారు", "ఉన్నార్", "ఉన్నాను",
             "ఉన్నాము", "ఉన్నాం", "ఉన్నావు", "ఉన్నావ్", "ఉన్నాయి", "ఉన్నాయ్", "ఉన్నవి", "ఉంద్"}
GRANTHIKA_EXIST = {"కద్దు", "కలదు", "కలవు", "కలరు", "కలడు", "గలదు", "గలవు", "గలరు", "గలడు"}
NEG_EQ = {"కాదు", "కాను", "కావు", "కాము", "కాం", "కాడు", "కావ్", "కారు"}
NEG_EXIST = {"లేదు", "లేను", "లేవు", "లేము", "లేం", "లేడు", "లేరు", "లేవ్", "లేద్"}
LAST_ONLY = {"లేదా", "ఉన్నా", "కారు"}
NOT_COPULA = {"కానీ", "కాని", "కానా", "కాకు", "కాక", "లేక", "లేకుండా", "కాకుండా"}

NONPRESENT_COPULA = {"ఉండేది", "ఉండేవాడు", "ఉండేవారు", "ఉండేదాన్ని", "ఉండేవాణ్ణి", "ఉండెను", "ఉండిరి",
                     "ఉండెడిది", "ఉండెడి", "ఉంటుంది", "ఉంటాడు", "ఉంటారు", "ఉంటాను", "ఉంటావు", "ఉంటాయి",
                     "ఉంటాం", "ఉంటావ్", "ఉండును", "ఉండగలదు", "ఉండదు", "ఉండవు", "ఉండడు", "ఉండరు",
                     "అయ్యె", "ఆయె", "ఆయెను", "అయెను", "ఉండేను", "ఉండియుండెను"}

VERBAL_SUFFIXES = (
    "తున్నాను", "తున్నాడు", "తున్నారు", "తున్నావు", "తున్నావ్", "తున్నాం", "తున్నాము", "తున్నది",
    "తుంది", "తున్నా", "తున్నాయి", "చున్నది", "చున్నాడు", "చున్నారు", "చున్నాను", "చుండెను", "చుండె",
    "తాను", "తావు", "తావ్", "తాడు", "తాడ్", "తారు", "తార్", "తాము", "తాం", "తాయి", "స్తా",
    "టాను", "టావు", "టాడు", "టారు", "టాం", "టాయి", "టుంది",
    "చ్చాను", "చ్చాడు", "చ్చారు", "చ్చావు", "చ్చాం", "చ్చింది", "చ్చా", "శాను", "శాడు", "శారు",
    "శావు", "శాం", "సాను", "సాడు", "సావు", "ట్టాను", "ట్టాడు", "ట్టారు", "ట్టింది", "ట్టా",
    "న్నాను", "న్నాడు", "న్నారు", "న్నావు", "న్నాం", "ింది", "య్యింది", "యింది", "య్యాను", "య్యాడు",
    "య్యారు", "య్యా", "కున్నాను", "కున్నాడు", "కున్నారు", "కుంది",
    "ెను", "ినది", "ినాడు", "ినారు", "ిరి", "ెదను", "ెదవు", "ెదము", "ుదును", "య్యె", "ాయె",
    "వచ్చును", "వలెను", "గలను", "గలము", "గలవు", "గలదు",
    "ండి", "ండీ", "దాం", "దామా", "కూడదు", "కండి", "ాలి", "వద్దు", "కావాలి", "కావాల",
    "యదు", "రాదు", "పోదు", "ండదు", "కుదరదు", "తెలీదు", "తెలుసు", "కోవాలి",
)
VERBAL_SUFFIX_EXCEPTIONS = {
    "బండి", "గండి", "మండి", "గాలి", "జాలి", "తాళి", "కాలి", "ఖాలి", "కింది",
    "వాడు", "వీడు", "మగాడు", "మొగాడు", "వారు", "గారు", "సారు", "తయారు", "చిరునామా",
    "తాను", "గిరి", "సిరి", "కిరికిరి", "బాదాం", "పిచ్చా", "ఉండి"
}
NONVERB_WHITELIST = {"ఎక్కడ", "యెక్కడ", "ఇక్కడ", "అక్కడ", "ఇక్కడే", "అక్కడే", "ఎక్కడో", "ఏంటి", "ఏమిటి",
                     "ఎవరు", "ఎందుకు", "ఎలా", "ఏం", "ఏమి", "సార్", "బాబూ", "బాబు", "అండి", "అండీ", "కదా",
                     "ఓకే", "అవును", "మరి", "అయ్యో", "అమ్మా", "రా", "రే", "ఒరేయ్", "హలో"}

ADJECTIVE_STEMS = {
    "మంచి", "మంచిది", "మంచోడు", "మంచోడివి", "మంచివాడు", "మంచివాళ్ళు", "చెడ్డ", "చెడ్డది", "చెడ్డవాడు",
    "పెద్ద", "పెద్దది", "పెద్దవాడు", "చిన్న", "చిన్నది", "చిన్నవాడు", "కొత్త", "కొత్తది", "పాత", "పాతది",
    "అందమైన", "అందమైనది", "గొప్ప", "గొప్పది", "గొప్పవాడు", "తెలివైన", "తెలివైనవాడు", "పిచ్చి", "పిచ్చోడు",
    "వెర్రి", "సులువు", "సులభం", "కష్టం", "తేలిక", "ఎక్కువ", "తక్కువ", "వేడి", "చల్లని", "తియ్యని",
    "చేదు", "సరి", "సరైన", "సరైనది", "ఖరీదు", "చవక", "సిద్ధం", "తయారు", "నిజమే", "ఫ్రెష్", "పర్ఫెక్ట్",
    "గ్రేట్", "సూపర్", "హ్యాపీ", "ఓకే", "బెస్ట్", "ఫైన్", "క్యూట్", "బిజీ", "ఫ్రీ", "రెడీ", "సేఫ్",
    "కరెక్ట్", "రాంగ్", "హాట్", "కూల్", "స్మార్ట్", "హ్యాండ్సమ్", "బ్యూటిఫుల్", "నైస్", "స్పెషల్",
    "సీరియస్", "సింపుల్", "ఈజీ", "డేంజర్", "డేంజరస్", "ఇంపార్టెంట్", "లక్కీ", "క్లియర్", "వీక్",
    "స్ట్రాంగ్", "స్వీట్", "క్రేజీ", "లేట్", "బాగు", "బాగా", "బాగానే", "బా", "బాగో",
    "సంతోషం", "ఆనందం", "భయం", "పాపం", "ముఖ్యం", "అవసరం", "గుణవతి", "విద్యానిధి"}
LOCATIVE_SUFFIXES = ("లో", "లోనే", "లోనూ", "లోన", "మీద", "మీదే", "పైన", "పై", "దగ్గర", "దగ్గరే", "వద్ద",
                     "కింద", "క్రింద", "పక్కన", "ముందు", "వెనక", "వెనుక", "బయట", "లోపల", "ఇంట", "యింట")
LOCATIVE_WORDS = {"ఇక్కడ", "అక్కడ", "ఎక్కడ", "యెక్కడ", "ఇక్కడే", "అక్కడే", "ఎక్కడో", "ఎక్కడా",
                  "ఇంట్లో", "ఇంటిలో", "బయట", "లోపల", "ఇటు", "అటు", "ఎటు"}
DISCOURSE_TAIL = {"బాబూ", "బాబు", "సార్", "సారు", "రా", "రే", "అండి", "అండీ", "కదా", "మరి", "ఏంటి",
                  "ఏమిటి", "అమ్మా", "అమ్మ", "ఒరేయ్", "రో", "లే", "గా", "అయ్యా", "తల్లీ", "నాన్నా", "ఓకే"}

TELUGU_PUNCT = string.punctuation + "“”‘’«»…–—।॥|•·"
ZW = dict.fromkeys(map(ord, "\u200b\u200c\u200d\ufeff\u0c01"), None)

def norm_token(tok):
    """Strip punctuation / zero-width chars / arasunna and prothetic వ-/య- glide (వుంది -> ఉంది)."""
    t = tok.translate(ZW).strip(TELUGU_PUNCT)
    if len(t) > 2 and t[:2] in ("వు", "యు") and ("న్న" in t[:5] or "ంద" in t[:4] or "ంట" in t[:4]):
        t = "ఉ" + t[2:]
    return t

def _variants(t):
    """Undo honorific (-అండి), emphatic/question clitics (-ా, -ే, -ో), and -లే/-రా tags."""
    out = {t}
    for suf in ("అండీ", "అండి", "ండీ", "ండి", "లే", "రా", "గా"):
        if t.endswith(suf) and len(t) > len(suf) + 1:
            base = t[:-len(suf)]
            out |= {base, base + "ి", base + "ు"}
    if t and t[-1] in "ాేోీూ":
        base = t[:-1]
        out |= {base + "ి", base + "ు", base}
    return out

FUSED_EXIST = sorted(("ు" + f[1:] for f in EXIST_AFF), key=len, reverse=True)
ADJ_NEG_STEMS = {"బా", "బాగా", "బాగో", "బాగు", "సరిగా", "బాగుండ"}
NOUN_NEG_STEMS = {"ఏం", "ఏమీ", "ఏమి", "ఎవరూ", "ఏదీ", "ఇంకేం", "మరేం", "పని", "డబ్బు", "టైం", "టైమ్",
                  "అవసరం", "సమస్య", "దీపము", "ఇంట్లోనే", "ఇంట్లో", "ఇక్కడ", "అక్కడ"}

def copula_of(tok, is_last):
    """Return (copula_family, canonical_form) for a token, 'VERBAL' for verb+negative fusions, or None."""
    t = norm_token(tok)
    if not t:
        return None
    if t in LAST_ONLY and not is_last:
        return None
    if t in NOT_COPULA:
        return None
    if t in NONPRESENT_COPULA:
        return ("NONPRESENT", t)
    for c in _variants(t):
        if c in EXIST_AFF:
            return ("EXIST", c)
        if c in GRANTHIKA_EXIST and c == t:
            return ("EXIST", c)
        if c in NEG_EQ:
            return ("NEG_EQ", c)
        if c in NEG_EXIST:
            return ("NEG_EXIST", c)
    for c in _variants(t):
        for fused in FUSED_EXIST:
            if c.endswith(fused) and len(c) > len(fused):
                prev = c[:-len(fused)]
                if prev[-1] in "తచక" or (len(prev) > 2 and prev[-2] == "్" and not prev.endswith("వ్వ")) or (prev.endswith("ంట") and not prev.startswith(("ఇంట", "యింట"))):
                    return None
                return ("EXIST", "ఉ" + fused[1:])
        for neg in NEG_EXIST:
            if c.endswith(neg) and len(c) > len(neg):
                stem = c[:-len(neg)]
                if stem in ADJ_NEG_STEMS or stem.endswith("గా") or stem in NOUN_NEG_STEMS:
                    return ("NEG_EXIST", neg)
                return ("VERBAL", c)
        for neg in ("కాదు", "కాను", "కావు"):
            if c.endswith(neg) and len(c) > len(neg) and not c.endswith(("కూడదు", "తాకాదు")):
                return ("NEG_EQ", neg)
    return None

ENGLISH_FUNCTION_WORDS = {"ఇస్", "ఈజ్", "ద", "యువర్", "ఐ", "హావ్", "వాట్", "వాట్స్", "టు", "అండ్", "ఆఫ్",
                          "బికాజ్", "ఆర్", "యు", "మై", "వెరీ", "డోంట్", "బి", "సో", "దట్", "దిస్", "ఇట్", "ఇఫ్",
                          "బట్", "ఫర్", "విత్", "వాస్", "విల్", "కెన్", "నాట్", "వియ్", "హి", "షి", "దే", "గో"}
MODIFIER_WORDS = {"ఒక", "ఒక్క", "ఒకటి", "ఆ", "ఈ", "యీ", "ఏ", "నా", "నీ", "మా", "మీ", "మన", "వాళ్ళ", "వాడి",
                  "దాని", "అతని", "ఆమె", "ఎంత", "ఇంత", "అంత", "ఎలా", "ఇలా", "అలా", "కూడా", "ఏదో", "ఏమాత్రం",
                  "చాలా", "బాగా", "ఇంకా", "మరి", "కానీ", "కాని", "అని", "అంటే", "అయితే", "ఎందుకంటే",
                  "కాబట్టి", "గనక", "గనుక", "అండ్", "ఇక", "యిక", "ఒక్కసారి", "రేపు", "ఇప్పుడు", "అసలు",
                  "జస్ట్", "యాక్చువల్", "బికాజ్", "కొంచెం", "కొంచం", "ప్రతి", "గారి", "తరఫు"}
INCOMPLETE_HEAD_ENDINGS = ("కి", "కీ", "కు", "ని", "నీ", "ను", "తో", "గా", "గానే", "గ", "ించి", "ిన", "లేని",
                           "న్న", "ంట", "కుండా", "ాక", "ప్పుడు", "తే", "నుంచి", "నుండి", "వల్ల", "కోసం",
                           "గురించి", "యొక్క", "-", "లాగా", "లా", "ంటే")
COMPLETE_HEAD_EXCEPTIONS = {"పని", "రాణి", "బోణి", "నేను", "నువ్వు", "నేనే", "మంచి", "పిచ్చి", "ఇష్టం", "కాని", "ఏంటి", "ఏమిటి",
                            "అసహ్యం", "సరే", "ఇదే", "అదే", "నిజమే", "కదే", "మనమే", "నాకే", "నీకే", "వాడే",
                            "ఎవరు", "ఎవరి", "రత్నమే", "తిరునామమే", "ఒక్కడే", "పేరేమిటి", "లోకంలో"}

def is_incomplete(ntoks, upos):
    i = len(ntoks) - 1
    while i > 0 and ntoks[i] in DISCOURSE_TAIL:
        i -= 1
    head = ntoks[i]
    if head in COMPLETE_HEAD_EXCEPTIONS or head.endswith(("లో", "లోనే", "మీద", "దగ్గర", "డను", "డవు", "రాలను")):
        return False
    if head in MODIFIER_WORDS or head in ENGLISH_FUNCTION_WORDS or head.isdigit():
        return True
    if head.endswith(INCOMPLETE_HEAD_ENDINGS):
        return True
    return upos[i] in ("VERB", "AUX", "ADV", "CCONJ", "SCONJ", "ADP", "DET")

def is_verbal_token(t):
    if t in VERBAL_SUFFIX_EXCEPTIONS or t in NONVERB_WHITELIST:
        return False
    res = copula_of(t, True)
    if res is not None and res[0] not in ("VERBAL", "NONPRESENT"):
        return False
    return t.endswith(VERBAL_SUFFIXES) or t in NONPRESENT_COPULA

def classify_telugu_predicate(norm_toks, upos, cop_idx, cop_family, adj_before):
    if cop_idx is not None:
        head_i = cop_idx - 1
    else:
        head_i = len(norm_toks) - 1
        while head_i > 0 and norm_toks[head_i] in DISCOURSE_TAIL:
            head_i -= 1
    head = norm_toks[head_i] if head_i >= 0 else ""
    head_upos = upos[head_i] if 0 <= head_i < len(upos) else ""
    head_stripped = re.sub(r"(ది|వాడు|వాళ్ళు|వాడివి|నే|ే)$", "", head)

    if adj_before or head in ADJECTIVE_STEMS or head_stripped in ADJECTIVE_STEMS or head_upos == "ADJ" \
            or (cop_family in ("EXIST", "NEG_EXIST") and head.endswith(("గా", "గానే", "గ"))):
        return "Adjectival"
    if cop_family in ("EXIST", "NEG_EXIST"):
        return "Locative/Existential"
    if head in LOCATIVE_WORDS or head.endswith(LOCATIVE_SUFFIXES) or any(t in LOCATIVE_WORDS for t in norm_toks):
        return "Locative/Existential"
    return "Equational/Nominal"

def annotate_telugu(utterances, register, sub_registers, source_title, id_prefix, min_tokens=3):
    import stanza
    nlp = get_telugu_stanza_pipeline()
    candidates = []
    for raw, sub in zip(utterances, sub_registers):
        sentence = re.sub(r"\s+", " ", raw.translate(ZW)).strip()
        toks = [w for w in sentence.split() if w.strip(TELUGU_PUNCT)]
        if len(toks) < min_tokens:
            continue
        if not re.search(r"[\u0C00-\u0C7F]", sentence):
            continue
        candidates.append((sentence, toks, sub))

    print(f"  {len(candidates)} candidate clauses (>= {min_tokens} tokens). Running batched Stanza ('te')...")
    docs = nlp([stanza.Document([], text=c[0]) for c in candidates]) if candidates else []

    rows = []
    for (sentence, toks, sub), doc in zip(candidates, docs):
        ntoks = [norm_token(w) for w in toks]
        words = [w for s in doc.sentences for w in s.words]
        stanza_upos = {norm_token(w.text): w.upos for w in words}
        upos = [stanza_upos.get(t, "") for t in ntoks]
        root = next((w for w in words if w.deprel == "root"), None)

        cop_idx, cop_family, cop_form, verbal_reason = None, None, None, None
        for i, tok in enumerate(toks):
            res = copula_of(tok, is_last=(i == len(toks) - 1 or all(t in DISCOURSE_TAIL for t in ntoks[i + 1:])))
            if res is None:
                continue
            if res[0] in ("VERBAL", "NONPRESENT"):
                verbal_reason = verbal_reason or ("Non-present copula" if res[0] == "NONPRESENT" else "Verb + negative aux")
                continue
            cop_idx, cop_family, cop_form = i, res[0], res[1]

        for i, t in enumerate(ntoks):
            if i == cop_idx or verbal_reason:
                continue
            if is_verbal_token(t):
                verbal_reason = "Verbal suffix"
            elif upos[i] in ("VERB", "AUX") and t not in NONVERB_WHITELIST:
                rel_participle = t.endswith(("ిన", "న్న", "ే", "ని", "ంత")) and i < len(ntoks) - 1
                if not rel_participle and copula_of(t, True) is None:
                    verbal_reason = "Stanza VERB"
        if any(x in sentence for x in ("http", "www.", "Subtitles")) or re.match(r"^\(.*\d.*\)$", sentence):
            verbal_reason = "Metadata/Fragment"
        if verbal_reason is None and re.search(r"వల్ల\s?కాదు|బోధ\s?కాదు", sentence):
            verbal_reason = "Verbal idiom (వల్ల కాదు 'not possible')"
        if verbal_reason is None and sum(t in ENGLISH_FUNCTION_WORDS for t in ntoks) >= 2:
            verbal_reason = "English (transliterated)"
        if verbal_reason is None and cop_idx is None and is_incomplete(ntoks, upos):
            verbal_reason = "Incomplete fragment"

        adj_before = cop_idx is not None and cop_idx >= 1 and (
            ntoks[cop_idx - 1].endswith(("గా", "గానే", "లా", "లాగా", "లాగ")) or ntoks[cop_idx - 1] in ADJECTIVE_STEMS)
        if cop_idx is not None and ntoks[cop_idx].startswith(("బాగ", "బాలే", "బాగో")):
            adj_before = True

        pred = classify_telugu_predicate(ntoks, upos, cop_idx, cop_family, adj_before)
        polarity = "Negative" if cop_family in ("NEG_EQ", "NEG_EXIST") else "Affirmative"

        rows.append({
            "Sentence_ID": f"{id_prefix}_{len(rows) + 1:04d}",
            "Register": register,
            "Sub_Register": sub,
            "Source_Title": source_title,
            "Raw_Sentence": sentence,
            "Copula_Status": 1 if cop_family else 0,
            "Copula_Form": cop_form or "",
            "Copula_Family": {"EXIST": "ఉండు (existential)", "NEG_EQ": "కాదు (neg. equational)",
                              "NEG_EXIST": "లేదు (neg. existential)"}.get(cop_family, "ZERO"),
            "Polarity": polarity,
            "Predicate_Type": pred,
            "Is_Valid_NonVerbal": verbal_reason is None,
            "Audit_Classification": verbal_reason or "Valid Non-Verbal Copular Clause",
            "NLP_Root_Word_Notes": f"Root Word: {root.text if root else 'N/A'} ({root.upos if root else 'UNK'})",
        })
    return pd.DataFrame(rows)

print("Telugu copula inventory and Stanza annotation functions defined.")

Telugu copula inventory and Stanza annotation functions defined.


In [8]:
# ---------------------------------------------------------------
# Step 5 & Step 6 · Telugu Subtitle (ASR json3) & Literary Extraction
# ---------------------------------------------------------------
PAUSE_S = 0.9
CLOSING_TAGS = {"సార్", "సర్", "మేడం", "మేడమ్", "రా", "రే", "అండి", "అండీ", "బాబు", "బాబూ", "కదా", "కదండీ",
                "అమ్మా", "నాన్నా", "బావా", "అన్నా", "ఒరేయ్", "రేయ్", "బే", "మామ", "మావా", "తల్లీ"}

def json3_words(path):
    j = json.load(open(path, encoding="utf-8"))
    words, music = [], []
    for e in j["events"]:
        t0 = e.get("tStartMs", 0)
        for s in e.get("segs", []):
            w = s.get("utf8", "").strip()
            if not w:
                continue
            if w.startswith("["):
                if "సంగీతం" in w or "Music" in w:
                    music.append((t0, t0 + e.get("dDurationMs", 0)))
                continue
            words.append((t0 + s.get("tOffsetMs", 0), w))
    return words, music

def is_clause_final(tok):
    t = norm_token(tok)
    res = copula_of(tok, True)
    if res is not None and res[0] != "VERBAL":
        return True
    return is_verbal_token(t) or (res is not None)

def json3_to_utterances(path, music_pad_ms=1500):
    words, music = json3_words(path)
    in_music = lambda t: any(a - music_pad_ms <= t <= b + music_pad_ms for a, b in music)
    utts, cur, prev_t, dropped = [], [], None, 0
    for t, w in words:
        if in_music(t):
            dropped += 1
            if cur: utts.append(" ".join(cur)); cur = []
            prev_t = t
            continue
        if cur and prev_t is not None and (t - prev_t) / 1000 > PAUSE_S:
            utts.append(" ".join(cur)); cur = []
        cur.append(w)
        prev_t = t
        if w[-1] in ".?!" or is_clause_final(w) or norm_token(w) in CLOSING_TAGS:
            utts.append(" ".join(cur)); cur = []
    if cur:
        utts.append(" ".join(cur))
    print(f"  {len(words)} ASR words ({dropped} in music stretches dropped) -> {len(utts)} utterances")
    return utts

def process_telugu_subtitles(json3_path, output_csv_path, movie_title):
    print(f"Processing Telugu film '{movie_title}' ...")
    utts = json3_to_utterances(json3_path)
    df = annotate_telugu(utts, "Spoken Subtitle", ["Film Dialogue"] * len(utts), movie_title, "SUB_" + movie_title[:4].upper())
    df.to_csv(output_csv_path, index=False, encoding="utf-8-sig")
    return df

OPEN_Q, CLOSE_Q = "“‘«", "”’»"

def split_dialogue_narrative(paragraph):
    """Walk a paragraph character-by-character and cut into (text, is_dialogue) segments by quote state."""
    segs, buf, inside = [], [], False
    for ch in paragraph:
        if ch in OPEN_Q or (ch == '"' and not inside):
            if buf: segs.append(("".join(buf), inside)); buf = []
            inside = True
            continue
        if ch in CLOSE_Q or (ch == '"' and inside):
            if buf: segs.append(("".join(buf), inside)); buf = []
            inside = False
            continue
        buf.append(ch)
    if buf:
        segs.append(("".join(buf), inside))
    return [(t.strip(), d) for t, d in segs if t.strip()]

def process_telugu_literary_txt(txt_path, output_csv_path, story_title):
    from indicnlp.tokenize import sentence_tokenize
    print(f"Processing Telugu story '{story_title}' ...")
    with open(txt_path, encoding="utf-8") as f:
        paragraphs = [p.strip() for p in f.read().split("\n") if p.strip()]
    paragraphs = paragraphs[1:]  # skip title line
    utts, subs = [], []
    for para in paragraphs:
        for seg, is_dlg in split_dialogue_narrative(para):
            for s in sentence_tokenize.sentence_split(seg, lang="te"):
                for piece in re.split(r"[;]", s):
                    if piece.strip():
                        utts.append(piece.strip())
                        subs.append("Literary Dialogue" if is_dlg else "Literary Narrative")
    df = annotate_telugu(utts, "Literary", subs, story_title, "LIT_" + story_title[:4].upper())
    df.to_csv(output_csv_path, index=False, encoding="utf-8-sig")
    return df


# ---------------------------------------------------------------
# Step 7 · Compile Telugu 6 Sources or Load Pre-Compiled Datasets
# ---------------------------------------------------------------
telugu_auto_path   = os.path.join(TELUGU_DIR, "Telugu_copula_dataset_automatic.csv")
telugu_manual_path = os.path.join(TELUGU_DIR, "Telugu_copula_dataset_manual.csv")

if RUN_TELUGU_EXTRACTION:
    download_telugu_raw_sources(TELUGU_DIR)
    MOVIES = {
        "Ala_Vaikunthapurramuloo": os.path.join(TELUGU_DIR, "subtitles", "ala_vaikunthapurramuloo.te-orig.json3"),
        "Srirastu_Subhamastu":     os.path.join(TELUGU_DIR, "subtitles", "srirastu_subhamastu.te-orig.json3"),
        "Vivaha_Bhojanambu":       os.path.join(TELUGU_DIR, "subtitles", "vivaha_bhojanambu.te-orig.json3"),
    }
    STORIES = {
        "Diddubatu":    os.path.join(TELUGU_DIR, "stories", "diddubatu.txt"),
        "Mee_Peremiti": os.path.join(TELUGU_DIR, "stories", "mee_peremiti.txt"),
        "Metilda":      os.path.join(TELUGU_DIR, "stories", "metilda.txt"),
    }
    te_dfs = []
    for title, path in MOVIES.items():
        te_dfs.append(process_telugu_subtitles(path, os.path.join(TELUGU_DIR, f"subtitles_{title.lower()}.csv"), title))
    for title, path in STORIES.items():
        te_dfs.append(process_telugu_literary_txt(path, os.path.join(TELUGU_DIR, f"formal_{title.lower()}.csv"), title))
    te_master = pd.concat(te_dfs, ignore_index=True)
    te_clean = te_master[te_master["Is_Valid_NonVerbal"]].copy()
    te_clean.to_csv(telugu_auto_path, index=False, encoding="utf-8-sig")

df_te_auto   = pd.read_csv(telugu_auto_path, encoding="utf-8-sig")
df_te_manual = pd.read_csv(telugu_manual_path, encoding="utf-8-sig")

print("STEP 7 SUMMARY — Telugu Automatic Corpus (N = 759) & Manually Verified Gold Set (N = 116)")
te_summary = df_te_auto.groupby(["Register", "Sub_Register", "Source_Title"]).agg(
    Auto_Valid_Clauses=("Copula_Status_NLP", "count"),
    Auto_Copula_Present=("Copula_Status_NLP", lambda x: int((x == 1).sum())),
    Auto_Zero_Copula=("Copula_Status_NLP", lambda x: int((x == 0).sum())),
).reset_index()
te_gold_counts = df_te_manual.groupby(["Register", "Sub_Register", "Source_Title"]).agg(
    Manual_Gold_Clauses=("Copula_Status_NLP", "count"),
    Manual_Zero_Copula=("Copula_Status_NLP", lambda x: int((x == 0).sum())),
).reset_index()
display(pd.merge(te_summary, te_gold_counts, on=["Register", "Sub_Register", "Source_Title"]))

STEP 7 SUMMARY — Telugu Automatic Corpus (N = 759) & Manually Verified Gold Set (N = 116)


,Register,Sub_Register,Source_Title,Auto_Valid_Clauses,Auto_Copula_Present,Auto_Zero_Copula,Manual_Gold_Clauses,Manual_Zero_Copula
0,Literary,Literary Dialogue,Diddubatu,7,3,4,5,2
1,Literary,Literary Dialogue,Mee_Peremiti,32,14,18,24,12
2,Literary,Literary Dialogue,Metilda,11,1,10,7,6
3,Literary,Literary Narrative,Diddubatu,3,1,2,2,1
4,Literary,Literary Narrative,Mee_Peremiti,32,15,17,24,10
5,Literary,Literary Narrative,Metilda,8,3,5,5,3
6,Spoken Subtitle,Film Dialogue,Ala_Vaikunthapurramuloo,263,107,156,16,5
7,Spoken Subtitle,Film Dialogue,Srirastu_Subhamastu,189,84,105,19,5
8,Spoken Subtitle,Film Dialogue,Vivaha_Bhojanambu,214,73,141,14,11


---
# PART III · Summary of All 5 Language Datasets Ready for Analysis

All five South Asian language corpora in `data/` are now compiled and ready for **Notebook 2 (`02_cross_linguistic_analysis.ipynb`)**:
1. **Hindi** (`data/hindi/Cross_register.csv` + `data/hindi/verification/verification.csv`)
2. **Assamese** (`data/assamese/formal_assamese.csv` + `data/assamese/informal_assamese.csv`)
3. **Bengali** (`data/bengali/Bengali_Cross_Register_Nonverbal.csv`)
4. **Telugu** (`data/telugu/Telugu_copula_dataset_automatic.csv` + `data/telugu/Telugu_copula_dataset_manual.csv`)
5. **Tamil** (`data/tamil/formal_tamil.csv` + `data/tamil/informal_tamil.csv`)

In [9]:
# ---------------------------------------------------------------
# Step 8 · Verify all 5 language datasets exist in data/
# ---------------------------------------------------------------
dataset_overview = [
    {"Order": 1, "Language": "Hindi (Central Indo-Aryan)",       "Data_Files": "data/hindi/Cross_register.csv (+ verification.csv)",                           "Raw_Rows": len(master_corpus), "Sources": "3 films (3 Idiots, Secret Superstar, Taare Zameen Par) + 3 stories (Idgah, Do Bailon Ki Katha, Pahalwan Ki Dholak)"},
    {"Order": 2, "Language": "Assamese (Eastern Indo-Aryan)",    "Data_Files": "data/assamese/formal_assamese.csv + informal_assamese.csv",                    "Raw_Rows": 50 + 51,            "Sources": "Paired Formal Prose essays (50) vs Informal Dialogue utterances (51)"},
    {"Order": 3, "Language": "Bengali (Eastern Indo-Aryan)",     "Data_Files": "data/bengali/Bengali_Cross_Register_Nonverbal.csv",                            "Raw_Rows": len(pd.read_csv(os.path.join(BASE_DIR, 'data', 'bengali', 'Bengali_Cross_Register_Nonverbal.csv'))), "Sources": "Satyajit Ray film (Goopy Gyne Bagha Byne) + 6 classic literary stories/essays"},
    {"Order": 4, "Language": "Telugu (South-Central Dravidian)", "Data_Files": "data/telugu/Telugu_copula_dataset_automatic.csv + Telugu_copula_dataset_manual.csv", "Raw_Rows": len(df_te_auto),    "Sources": "3 YouTube ASR films + 3 Gurajada Apparao short stories (759 auto / 116 manual gold)"},
    {"Order": 5, "Language": "Tamil (South Dravidian)",          "Data_Files": "data/tamil/formal_tamil.csv + informal_tamil.csv",                             "Raw_Rows": 53 + 53,            "Sources": "Paired Formal Written Prose (53) vs Informal Spoken Colloquial (53)"},
]
display(pd.DataFrame(dataset_overview))

,Order,Language,Data_Files,Raw_Rows,Sources
0,1,Hindi (Central Indo-Aryan),data/hindi/Cross_register.csv (+ verification....,4334,"3 films (3 Idiots, Secret Superstar, Taare Zam..."
1,2,Assamese (Eastern Indo-Aryan),data/assamese/formal_assamese.csv + informal_a...,101,Paired Formal Prose essays (50) vs Informal Di...
2,3,Bengali (Eastern Indo-Aryan),data/bengali/Bengali_Cross_Register_Nonverbal.csv,373,Satyajit Ray film (Goopy Gyne Bagha Byne) + 6 ...
3,4,Telugu (South-Central Dravidian),data/telugu/Telugu_copula_dataset_automatic.cs...,759,3 YouTube ASR films + 3 Gurajada Apparao short...
4,5,Tamil (South Dravidian),data/tamil/formal_tamil.csv + informal_tamil.csv,106,Paired Formal Written Prose (53) vs Informal S...
